# Capstone — Refresh / Content Opportunity Scoring

This notebook mirrors the public paper with measured decision-support claims.

## 1. Question

Can an interpretable score prioritize sufficiently visible pages so the top-50 queue contains more observed decline cases than a transparent baseline? The decision-maker is an SEO editor allocating human review.

In [1]:
print("Question: rank current decline-associated pages for human review.")
print("Unit: one pseudonymized content page; action: prioritize review.")

Question: rank current decline-associated pages for human review.
Unit: one pseudonymized content page; action: prioritize review.


## 2. Data

The authorized starter snapshot has 30,000 pages and 32 pseudonymous clients. The label proxy is trend_direction == "down"; it is current-state, not future. Five decision-time numeric fields are used. IDs are grouping/output only; trend_direction and trend_pct are excluded.

In [2]:
import pandas as pd
from pathlib import Path
root=Path.cwd()
if not (root/"data/processed/refresh_feature_vector.csv").exists(): root=(root/"../..").resolve()
frame=pd.read_csv(root/"data/processed/refresh_feature_vector.csv")
features=["impressions_90d","days_since_last_update","avg_position","ctr","engagement_rate"]
print(frame.shape, frame.client_id.nunique(), features)
assert len(frame)==30000 and frame.client_id.nunique()==32

(30000, 52) 32 ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'engagement_rate']


## 3. Methodology

The baseline combines visibility, freshness, position opportunity, and depth signals. Candidate models were logistic regression, decision tree, and random forest. ML-09 selected the decision tree inside an inner client-grouped split and evaluated it on a disjoint outer client holdout with train-only preprocessing. Precision@50 is the share of observed decline labels in the top 50.

In [3]:
print("Validation: nested client-grouped split; outer client overlap = 0.")
print("Leakage controls: no IDs, trend_direction, trend_pct, or future fields as features.")

Validation: nested client-grouped split; outer client overlap = 0.
Leakage controls: no IDs, trend_direction, trend_pct, or future fields as features.


## 4. Results (vs baseline)

The audited outer holdout measured baseline Precision@50 0.280 and decision-tree Precision@50 0.540 (absolute difference +0.260). This is directional evidence from one retrospective evaluation, not statistical significance or causal impact.

In [4]:
results=pd.DataFrame([{"method":"audited baseline","precision_at_50":0.280},{"method":"audited decision tree","precision_at_50":0.540}]); display(results); assert results.precision_at_50.tolist()==[0.28,0.54]

               method  precision_at_50
     audited baseline             0.28
audited decision tree             0.54


## 5. Limitations

Current decline is not a future outcome. Results may vary by client and time; the original ML-08 holdout had selection bias. Ranking quality does not prove a refresh improves SEO. Human review, prospective measurement, and approval remain required; no ROI or traffic uplift was measured.

In [5]:
print("Honest framing: observed, measured, directional, decision-support; no causal or production claim.")

Honest framing: observed, measured, directional, decision-support; no causal or production claim.


## 6. Ranked recommendations

ML-10 exports scores, reason codes, actions, evidence notes, and mandatory review status. Editors check relevance, completeness, intent, seasonality, cost, and approvals; no-go cases are escalated.

In [6]:
out=root/"work/outputs/content_action_playbook.csv"; q=pd.read_csv(out); print(q.shape); display(q.head(3)); assert len(q)==30000 and q.content_id.is_unique

(30000, 8)
 priority_rank           content_id  action_score          reason_code  recommended_action                         evidence_summary                                                      confidence_note human_review_status
             1 content_00821613c11f      0.773631      LOW_CTR_VISIBLE   REVIEW_TITLE_META       Visible page has below-median CTR. Ranks association with current decline proxy; not a future forecast.            REQUIRED
             2 content_0255d506d5f2      0.773631 POSITION_OPPORTUNITY INVESTIGATE_RANKING Visible page is below page-one position. Ranks association with current decline proxy; not a future forecast.            REQUIRED
             3 content_02ae9b37d7b7      0.773631      LOW_CTR_VISIBLE   REVIEW_TITLE_META       Visible page has below-median CTR. Ranks association with current decline proxy; not a future forecast.            REQUIRED


## 7. Artifacts the paper embeds

The paper includes the audited comparison table/chart and links to this notebook, ML-09 validation, and the ML-10 playbook.

In [7]:
print("Paper artifacts: results table, comparison bars, ranked recommendation CSV, reproducibility links.")

Paper artifacts: results table, comparison bars, ranked recommendation CSV, reproducibility links.


## Self-check

- [x] Sections mirror the public paper.
- [x] Metrics match ML-09 audited outputs.
- [x] Leakage and client grouping are explicit.
- [x] ML-12 deliverables are in work/.
- [x] Claims are decision-support, not causal.